In [ ]:
import pandas as pd

# 파일 경로 설정 (다운로드한 폴더 기준)

# ratings.dat 로드
ratings = pd.read_csv(ratings_path, sep="::", engine='python',
                      names=['UserID', 'MovieID', 'Rating', 'Timestamp'])

# movies.dat 로드
movies = pd.read_csv(movies_path, sep="::", engine='python',
                     names=['MovieID', 'Title', 'Genres'])

# 데이터 확인
print(ratings.head())
print(movies.head())

   UserID  MovieID  Rating  Timestamp
0       1      122     5.0  838985046
1       1      185     5.0  838983525
2       1      231     5.0  838983392
3       1      292     5.0  838983421
4       1      316     5.0  838983392
   MovieID                               Title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        Genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  


In [14]:
ratings

,UserID,MovieID,Rating,Timestamp
0,1,122,5.0,838985046
1,1,185,5.0,838983525
2,1,231,5.0,838983392
3,1,292,5.0,838983421
4,1,316,5.0,838983392
...,...,...,...,...
10000049,71567,2107,1.0,912580553
10000050,71567,2126,2.0,912649143
10000051,71567,2294,5.0,912577968
10000052,71567,2338,2.0,912578016


In [15]:
movies

,MovieID,Title,Genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
10676,65088,Bedtime Stories (2008),Adventure|Children|Comedy
10677,65091,Manhattan Melodrama (1934),Crime|Drama|Romance
10678,65126,Choke (2008),Comedy|Drama
10679,65130,Revolutionary Road (2008),Drama|Romance


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from sklearn.model_selection import train_test_split

# Device 설정 (CUDA 사용 가능 시)
device = torch.device('mps' if torch.mps.is_available() else 'cpu')
print(device)
# MovieLens 10M ratings 데이터 로드
ratings = pd.read_csv(ratings_path, sep="::", engine='python',
                      names=['UserID', 'MovieID', 'Rating', 'Timestamp'])

# ID를 0부터 시작하는 인덱스로 변환 (Embedding을 위해)
user2idx = {id: idx for idx, id in enumerate(ratings['UserID'].unique())}
movie2idx = {id: idx for idx, id in enumerate(ratings['MovieID'].unique())}

ratings['UserID'] = ratings['UserID'].map(user2idx)
ratings['MovieID'] = ratings['MovieID'].map(movie2idx)

# Train/Test Split
train_data, test_data = train_test_split(ratings, test_size=0.2, random_state=42)

# PyTorch Dataset
class MovieLensDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.users = torch.tensor(df['UserID'].values, dtype=torch.long)
        self.movies = torch.tensor(df['MovieID'].values, dtype=torch.long)
        self.ratings = torch.tensor(df['Rating'].values, dtype=torch.float32)

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        return self.users[idx], self.movies[idx], self.ratings[idx]

train_dataset = MovieLensDataset(train_data)
test_dataset = MovieLensDataset(test_data)


# DataLoader 설정
# train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=1024, shuffle=True)
# test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1024)

mps


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import time
import psutil
import os
from torch.utils.data import DataLoader

import multiprocessing


def main():
    num_cores = multiprocessing.cpu_count()
    print(f"Available CPU cores: {num_cores}")

    # MPS 디바이스 설정
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    print(f"Using device: {device}")

    # 현재 프로세스 메모리 체크 함수 (MB 단위)
    def get_memory_usage():
        process = psutil.Process(os.getpid())
        mem = process.memory_info().rss / 1024 ** 2  # in MB
        return mem

    # Matrix Factorization 모델
    class MatrixFactorization(nn.Module):
        def __init__(self, num_users, num_items, emb_size=50):
            super().__init__()
            self.user_emb = nn.Embedding(num_users, emb_size)
            self.item_emb = nn.Embedding(num_items, emb_size)

        def forward(self, user_ids, item_ids):
            user_vecs = self.user_emb(user_ids)
            item_vecs = self.item_emb(item_ids)
            return (user_vecs * item_vecs).sum(1)

    # 유저, 아이템 수
    num_users = len(user2idx)
    num_items = len(movie2idx)

    # 실험할 batch size 목록
    batch_sizes = [64, 128, 256, 512, 1024, 2048, 4096]
    results = []

    criterion = nn.MSELoss()

    for batch_size in batch_sizes:
        print(f"\nTrying batch size: {batch_size}")
        try:
            train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,num_workers=num_cores//2)

            model = MatrixFactorization(num_users, num_items).to(device)
            optimizer = optim.Adam(model.parameters(), lr=0.005)

            model.train()
            start_time = time.time()
            mem_before = get_memory_usage()

            for users, items, ratings in train_loader:
                users = users.to(device)
                items = items.to(device)
                ratings = ratings.to(device)

                optimizer.zero_grad()
                outputs = model(users, items)
                loss = criterion(outputs, ratings)
                loss.backward()
                optimizer.step()

            torch.mps.synchronize()  # 동기화
            mem_after = get_memory_usage()
            end_time = time.time()

            elapsed = end_time - start_time
            mem_used = mem_after - mem_before

            print(f"Batch size {batch_size}: Time = {elapsed:.2f}s | Memory Used = {mem_used:.2f} MB")
            results.append((batch_size, elapsed, mem_used))

        except RuntimeError as e:
            print(f"Batch size {batch_size} failed: {e}")

    # 최적 결과 출력
    if results:
        results.sort(key=lambda x: x[1])  # 시간 기준 정렬
        print("\n=== Summary ===")
        for b, t, m in results:
            print(f"Batch Size: {b} | Time: {t:.2f}s | Mem: {m:.2f}MB")
        best = results[0]
        print(f"\nBest Batch Size: {best[0]} (Time: {best[1]:.2f}s, Mem: {best[2]:.2f}MB)")
    else:
        print("No batch size succeeded.")


if __name__ == "__main__":
    main()


Available CPU cores: 10
Using device: mps

Trying batch size: 64


PicklingError: Can't pickle <class '__main__.MovieLensDataset'>: it's not the same object as __main__.MovieLensDataset

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import pandas as pd
from sklearn.model_selection import train_test_split

# 사용할 디바이스 설정 (Apple Silicon GPU 가속인 mps 사용 가능 시 사용)
device = torch.device('mps' if torch.mps.is_available() else 'cpu')

# 데이터 로딩 및 전처리
ratings = pd.read_csv(ratings_path, sep="::", engine='python',  # 데이터셋 로드 (::로 구분된 csv 파일)
                      names=['UserID', 'MovieID', 'Rating', 'Timestamp'])  # 컬럼명 지정

# 유저와 영화 ID를 인덱스로 매핑 (0부터 시작하는 연속된 인덱스로 변환)
user2idx = {id: idx for idx, id in enumerate(ratings['UserID'].unique())}
movie2idx = {id: idx for idx, id in enumerate(ratings['MovieID'].unique())}

# 매핑된 인덱스를 데이터프레임에 적용
ratings['UserID'] = ratings['UserID'].map(user2idx)
ratings['MovieID'] = ratings['MovieID'].map(movie2idx)

# train/test 데이터 분할
train_data, test_data = train_test_split(ratings, test_size=0.2, random_state=42)

# 커스텀 Dataset 클래스 정의
class MovieLensDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.users = torch.tensor(df['UserID'].values, dtype=torch.long)  # 사용자 ID
        self.movies = torch.tensor(df['MovieID'].values, dtype=torch.long)  # 영화 ID
        self.ratings = torch.tensor(df['Rating'].values, dtype=torch.float32)  # 평점

    def __len__(self):
        return len(self.users)  # 데이터 길이 반환

    def __getitem__(self, idx):
        return self.users[idx], self.movies[idx], self.ratings[idx]  # 인덱스에 해당하는 데이터 반환

# 학습 데이터에서 1% 샘플링
sample_ratio = 0.01
train_data_sampled = train_data.sample(frac=sample_ratio, random_state=42)
test_data_sampled = test_data.sample(frac=sample_ratio, random_state=42)

# Dataset 객체 생성
train_dataset = MovieLensDataset(train_data_sampled)
test_dataset = MovieLensDataset(test_data_sampled)

# DataLoader 생성 (batch_size=2048, 학습 데이터는 셔플)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=2048, shuffle=True, pin_memory=False)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=2048)

### Mixture of Experts (MoE) + Top-k + Dropout 모델 정의
class MoE_TopK_Dropout(nn.Module):
    def __init__(self, num_users, num_movies, embedding_dim=50, num_experts=4, top_k=2, expert_dropout=0.2):
        super(MoE_TopK_Dropout, self).__init__()
        self.num_experts = num_experts  # 전문가 수
        self.top_k = top_k  # top-k 전문가 선택
        self.expert_dropout = expert_dropout  # 전문가 dropout 비율

        # 유저와 영화 임베딩 정의
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.movie_embedding = nn.Embedding(num_movies, embedding_dim)

        # Gating Network 정의 (user + movie embedding을 받아 expert score 출력)
        self.gate = nn.Linear(embedding_dim * 2, num_experts)

        # 여러 expert 네트워크 정의 (ModuleList 사용)
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(embedding_dim * 2, 128),  # 첫 번째 FC layer
                nn.ReLU(),  # ReLU 활성화 함수
                nn.Linear(128, 64),  # 두 번째 FC layer
                nn.ReLU(),  # ReLU 활성화 함수
                nn.Linear(64, 1)  # 최종 출력 (rating 예측)
            ) for _ in range(num_experts)
        ])

    def forward(self, user_ids, movie_ids):
        # 유저와 영화 ID를 임베딩 벡터로 변환
        user_embed = self.user_embedding(user_ids)  # [batch_size, embedding_dim]
        movie_embed = self.movie_embedding(movie_ids)  # [batch_size, embedding_dim]
        x = torch.cat([user_embed, movie_embed], dim=-1)  # [batch_size, embedding_dim*2]

        # Gating 네트워크에서 전문가별 점수 계산 (softmax 적용 안함)
        gate_logits = self.gate(x)  # [batch_size, num_experts]

        # Top-k 전문가 선택 (가장 점수가 높은 k개의 expert 선택)
        topk_values, topk_indices = torch.topk(gate_logits, self.top_k, dim=-1)

        # 선택된 top-k 전문가에 대해 softmax로 weight 부여
        topk_weights = F.softmax(topk_values, dim=-1)  # [batch_size, k]

        outputs = []
        for i in range(self.top_k):
            idx = topk_indices[:, i]  # 현재 선택된 expert의 인덱스 [batch_size]
            weight = topk_weights[:, i]  # 선택된 expert에 대한 weight [batch_size]

            # expert dropout mask 생성 및 적용
            mask = (torch.rand_like(weight) > self.expert_dropout).float()
            weight = weight * mask  # dropout 반영된 weight

            # 각 샘플마다 해당 expert를 선택해서 실행
            expert_out = torch.stack([
                self.experts[expert_id](x[j].unsqueeze(0)).squeeze()  # 각 expert에 대해 forward pass
                for j, expert_id in enumerate(idx)
            ])
            outputs.append(expert_out * weight)  # weight 적용 후 outputs에 추가

        # 선택된 k개의 expert 출력값을 합산
        final_output = torch.stack(outputs, dim=0).sum(dim=0)  # [batch_size]
        return final_output

# 모델 초기화 (유저 수, 영화 수를 인자로 넘김)
num_users = len(user2idx)
num_movies = len(movie2idx)
model = MoE_TopK_Dropout(num_users, num_movies, num_experts=4, top_k=2, expert_dropout=0.2).to(device)

# 손실 함수 및 최적화 함수 정의 (MSE + Adam)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 학습 루프 (5 epoch)
for epoch in range(5):
    model.train()  # 학습 모드
    total_loss = 0  # epoch별 총 손실 초기화
    for user_ids, movie_ids, ratings in train_loader:
        # 데이터를 디바이스로 이동
        user_ids, movie_ids, ratings = user_ids.to(device), movie_ids.to(device), ratings.to(device)

        optimizer.zero_grad()  # 기울기 초기화
        outputs = model(user_ids, movie_ids)  # 모델 forward pass
        loss = criterion(outputs, ratings)  # 손실 계산
        loss.backward()  # 역전파
        optimizer.step()  # 파라미터 업데이트
        total_loss += loss.item()  # 손실 누적

    print(f'Epoch [{epoch+1}/5], Loss: {total_loss/len(train_loader):.4f}')  # epoch별 평균 손실 출력

# 테스트 데이터셋으로 평가
model.eval()  # 평가 모드
with torch.no_grad():  # 기울기 계산 비활성화
    total_loss = 0  # 총 손실 초기화
    for user_ids, movie_ids, ratings in test_loader:
        user_ids, movie_ids, ratings = user_ids.to(device), movie_ids.to(device), ratings.to(device)
        outputs = model(user_ids, movie_ids)  # 모델 forward pass
        loss = criterion(outputs, ratings)  # 손실 계산
        total_loss += loss.item()  # 손실 누적
    print(f'Test Loss (MSE): {total_loss/len(test_loader):.4f}')  # 테스트 평균 손실 출력


Epoch [1/5], Loss: 6.4389
Test Loss (MSE): 2.9175


In [19]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

def evaluate_model(model, test_loader):
    model.eval()
    preds = []
    trues = []
    with torch.no_grad():
        for user_ids, movie_ids, ratings in test_loader:
            user_ids, movie_ids, ratings = user_ids.to(device), movie_ids.to(device), ratings.to(device)
            outputs = model(user_ids, movie_ids)
            preds.extend(outputs.cpu().numpy())
            trues.extend(ratings.cpu().numpy())

    preds = np.clip(preds, 0.5, 5.0)  # 평점 범위 조정 (MovieLens는 0.5 ~ 5.0)
    rmse = np.sqrt(mean_squared_error(trues, preds))
    mae = mean_absolute_error(trues, preds)

    print(f"✅ RMSE: {rmse:.4f}")
    print(f"✅ MAE: {mae:.4f}")

# 평가 실행
evaluate_model(model, test_loader)


✅ RMSE: 1.6522
✅ MAE: 1.3032


In [20]:
def recommend_top_n(model, user_id, n=5):
    model.eval()
    user_idx = user2idx[user_id]
    user_tensor = torch.tensor([user_idx] * num_movies, dtype=torch.long).to(device)
    movie_tensor = torch.tensor(list(range(num_movies)), dtype=torch.long).to(device)

    with torch.no_grad():
        preds = model(user_tensor, movie_tensor)

    preds = preds.cpu().numpy()
    top_n_idx = preds.argsort()[-n:][::-1]  # 높은 점수 순으로 N개

    idx2movie = {idx: movie for movie, idx in movie2idx.items()}
    recommended_movies = [idx2movie[i] for i in top_n_idx]

    print(f"🎯 User {user_id}에게 추천하는 Top-{n} 영화 ID: {recommended_movies}")

# 예시: user1에게 추천
recommend_top_n(model, user_id=list(user2idx.keys())[0], n=5)


🎯 User 1에게 추천하는 Top-5 영화 ID: [7335, 2690, 7458, 8982, 318]


In [17]:
user2idx[1]

0

In [16]:
num_movies

10677